---
title: "Stage 0: Research Experiment"
author: "Abdelhamed Nour"
description: "Training Pascal VOC segmentation directly in a notebook"
format:
  html:
    toc: true
    code-fold: false
    code-tools: true
execute:
  eval: false
  echo: true
---


## Starting Point

A computer vision student usually starts with the most useful question:

**Can I make the experiment train?**

This notebook keeps the code close to the analysis so it is easy to inspect images, masks, losses, and mistakes.

::: {.notes}
Timing: 3 minutes. Present this as a valid research workflow, not as bad code. The tradeoff is speed of thought over long-term team reliability.
:::

## Imports And Local Choices

The first version keeps experiment choices directly in the notebook.

In [4]:
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import VOCSegmentation
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF
import segmentation_models_pytorch as smp

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "params.yaml").exists() else Path.cwd().parent
DATA_ROOT = PROJECT_ROOT / "data"
IMAGE_SIZE = 128
MAX_SAMPLES = 12
BATCH_SIZE = 2
NUM_CLASSES = 21
DEVICE = "cpu"

## VOC Dataset Setup

Use the real Pascal VOC segmentation dataset, but restrict the run to a tiny subset for a live CPU demo.

In [5]:
def voc_transforms(image, mask):
    image = image.convert("RGB")
    image = TF.resize(
        image,
        [IMAGE_SIZE, IMAGE_SIZE],
        interpolation=InterpolationMode.BILINEAR,
    )
    mask = TF.resize(
        mask,
        [IMAGE_SIZE, IMAGE_SIZE],
        interpolation=InterpolationMode.NEAREST,
    )
    image = TF.to_tensor(image)
    mask = torch.as_tensor(np.array(mask), dtype=torch.long)
    return image, mask

train_ds = VOCSegmentation(
    root=DATA_ROOT,
    year="2012",
    image_set="train",
    download=True,
    transforms=voc_transforms,
)
train_ds = Subset(train_ds, range(min(MAX_SAMPLES, len(train_ds))))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

images, masks = next(iter(train_loader))
images.shape, masks.shape

 34%|██████████████████████████████▌                                                            | 671M/2.00G [18:09<35:56, 616kB/s]


RuntimeError: File not found or corrupted.

## Model And One Training Step

This proves the wiring: image tensor, target mask, model output, loss, and optimizer step.

In [ ]:
def sanitize_mask(mask, num_classes=NUM_CLASSES):
    mask = mask.clone()
    invalid = (mask < 0) | ((mask >= num_classes) & (mask != 255))
    mask[invalid] = 255
    return mask

model = smp.Unet(
    encoder_name="resnet18",
    encoder_weights=None,
    in_channels=3,
    classes=NUM_CLASSES,
).to(DEVICE)

loss_fn = torch.nn.CrossEntropyLoss(ignore_index=255)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

model.train()
images = images.to(DEVICE)
masks = sanitize_mask(masks.to(DEVICE))

optimizer.zero_grad(set_to_none=True)
logits = model(images)
loss = loss_fn(logits, masks)
loss.backward()
optimizer.step()

{"loss": float(loss.detach().cpu()), "logits_shape": tuple(logits.shape)}

## What This Stage Gives Us

- Fast feedback
- Visual inspection
- A concrete semantic segmentation task
- A working baseline for discussion

## What It Does Not Give Us Yet

- A clear way to change experiments
- A stable entry point for another person
- A testable contract
- Run history

## The Pressure To Change

```{mermaid}
%%| eval: true
%%| echo: false
flowchart LR
  A[Research notebook] --> B[Works on my machine]
  B --> C{Can someone else change it?}
  C -->|Not safely yet| D[Extract the workflow]
```

::: {.notes}
Transition to the second notebook: another researcher joins and asks to train a slightly different configuration.
:::